# 03 - Silver Listings

## Objetivo

Realizar a transformação e padronização da tabela `listings`,
proveniente da camada Bronze, preparando os dados para análises
de desempenho dos imóveis e posterior avaliação de potencial
de investimento em imóveis para aluguel de curta duração em
João Pessoa - PB.

## Fonte

Tabela de origem:

`airbnb_joao_pessoa.bronze.listings`

## Responsabilidades

Nesta etapa serão realizadas:

- avaliação da qualidade dos dados;
- verificação de duplicidades;
- tratamento de valores nulos;
- padronização de tipos de dados;
- padronização de campos textuais;
- validação dos domínios das variáveis;
- tratamento de valores inconsistentes;
- criação de atributos derivados relevantes para análise;
- organização das colunas para a camada analítica.

## Princípio de transformação

A camada Silver tem como objetivo disponibilizar dados
consistentes, padronizados e confiáveis para as etapas
analíticas posteriores.

As transformações serão aplicadas com base na estrutura e
qualidade observadas na camada Bronze, evitando a remoção
de informações sem justificativa.

Os dados originais permanecem preservados na camada Bronze.

## Tabela de saída

`airbnb_joao_pessoa.silver.listings`

In [0]:
# Leitura
listings = spark.table("airbnb_joao_pessoa.bronze.listings")

display(listings)

In [0]:
print(f"Quantidade de registros: {listings.count()}")
print(f"Quantidade de colunas: {len(listings.columns)}")

In [0]:
from pyspark.sql.functions import sum, when, col

null_summary = (
    listings
    .select([
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in listings.columns
    ])
)

display(null_summary)

Conforme validação do notebook anterior:

| Variável                  | Tipo     | Estratégia inicial                            |
| ------------------------- | -------- | --------------------------------------------- |
| `description`             | texto    | manter nulo                                   |
| `photo_urls`              | texto    | manter nulo                                   |
| `host_name`               | texto    | manter nulo                                   |
| `guests`                  | numérico | investigar antes de preencher                 |
| `bedrooms`                | numérico | investigar antes de preencher                 |
| `registration`            | boolean  | manter nulo / criar flag                      |
| `instant_book`            | boolean  | manter nulo / criar flag                      |
| `professional_management` | boolean  | manter nulo / criar flag                      |
| `checkin_time`            | texto    | manter nulo                                   |
| `checkout_time`           | texto    | manter nulo                                   |
| `guest_favorite`          | boolean  | manter nulo                                   |
| `exact_location`          | boolean  | manter nulo                                   |
| `cleaning_fee`            | numérico | **não transformar nulo em 0 automaticamente** |
| `extra_guest_fee`         | numérico | **não transformar nulo em 0 automaticamente** |
| `single_fee_structure`    | boolean  | manter nulo                                   |


In [0]:
# Padronizando strings
from pyspark.sql.functions import col, trim
text_columns = [
    "listing_name", "listing_type", "room_type", "host_name", "description", "photo_urls", "cancellation_policy",
    "checkin_time", "checkout_time", "currency"
]

for column_name in text_columns:
    listings = listings.withColumn(
        column_name,
        trim(col(column_name))
    )

In [0]:
# padronizando variáveis categóricas

display(listings.select("listing_type").distinct().orderBy("listing_type"))
display(listings.select("room_type").distinct().orderBy("room_type"))
display(listings.select("cancellation_policy").distinct().orderBy("cancellation_policy"))


In [0]:
# invesitgando os nulos de variáveis importantes para a análise

display(
    listings
    .filter(col("guests").isNull() | col("bedrooms").isNull() | col("beds").isNull() | col("baths").isNull())
    .select("listing_id", "listing_name", "guests", "bedrooms", "beds", "baths", "room_type", "listing_type")
)

In [0]:
display(listings.select("guests", "bedrooms", "beds", "baths").summary())

In [0]:
# com relação as taxas

display(
    listings.select("listing_id", "guests", "cleaning_fee", "extra_guest_fee", "single_fee_structure")
    .orderBy("listing_id")
)

In [0]:
display(listings.select("cleaning_fee", "extra_guest_fee").summary())

In [0]:
# Criando indicadores de disponibilidade da informação
from pyspark.sql.functions import when

listings = listings.withColumn("has_description", when(col("description").isNotNull(), True).otherwise(False))
listings = listings.withColumn("has_photos", when(col("photo_urls").isNotNull(), True).otherwise(False))

In [0]:
# Validação de valores numéricos
display(
    listings.filter(
        (col("guests") < 0) | (col("bedrooms") < 0) | (col("beds") < 0) | (col("baths") < 0) |  (col("cleaning_fee") < 0) |
        (col("extra_guest_fee") < 0)
    )
)

In [0]:
rating_columns = ["rating_overall", "rating_accuracy", "rating_checkin", "rating_cleanliness", "rating_communication", "rating_location", "rating_value"]

for c in rating_columns:
    display(listings.filter((col(c) < 0) | (col(c) > 5)).select("listing_id", c))

In [0]:
display(
    listings.filter(
        (col("ttm_occupancy") < 0) | (col("ttm_occupancy") > 1) | (col("l90d_occupancy") < 0) | (col("l90d_occupancy") > 1)
    )
)

#### Criação de atributos derivados

In [0]:
# Imóvel inteiro ou quarto privado
listings = listings.withColumn(
    "property_usage_type",
    when(col("room_type") == "entire_home", "entire_property")
    .when(col("room_type") == "private_room", "private_room")
    .when(col("room_type") == "hotel_room", "hotel_room")
    .otherwise(None)
)

In [0]:
# Indicador de imóvel com informação de capacidade
listings = listings.withColumn(
    "capacity_info_available",
    when(
        col("guests").isNotNull() & col("bedrooms").isNotNull() & col("beds").isNotNull() & col("baths").isNotNull(),
        True
    ).otherwise(False)
)

In [0]:
# Identificador de informações de taxas
listings = listings.withColumn(
    "fee_info_available",
    when(col("cleaning_fee").isNotNull() & col("extra_guest_fee").isNotNull(), True).otherwise(False))

In [0]:
# Receita anual disponivel
listings = listings.withColumn("revenue_info_available", when(col("ttm_revenue").isNotNull(), True).otherwise(False))

## Validação da camada Silver

Após as transformações, serão realizadas validações para garantir:

- manutenção da quantidade de registros;
- unicidade de `listing_id`;
- ausência de valores inválidos;
- consistência dos domínios das variáveis;
- preservação dos valores nulos não imputados;
- consistência dos novos atributos derivados.

In [0]:
print(f"Quantidade de registros: {listings.count()}")
print(f"Quantidade de colunas: {len(listings.columns)}")

In [0]:
# validar duplicidade
duplicated_listings = (listings.groupBy("listing_id").count().filter(col("count") > 1))
display(duplicated_listings)

In [0]:
# validar capacidade
display(
    listings.filter((col("guests") < 0) | (col("bedrooms") < 0) | (col("beds") < 0) | (col("baths") < 0)
    ).select("listing_id", "guests", "bedrooms", "beds", "baths")
)

In [0]:
# validar taxas
display(
    listings.filter((col("cleaning_fee") < 0) | (col("extra_guest_fee") < 0)
    ).select("listing_id", "cleaning_fee", "extra_guest_fee")
)

In [0]:
# validas hatings
rating_columns = ["rating_overall", "rating_accuracy", "rating_checkin", "rating_cleanliness", "rating_communication",
    "rating_location", "rating_value"]

for c in rating_columns:
    invalid_ratings = listings.filter((col(c) < 0) | (col(c) > 5))
    print(f"{c}: {invalid_ratings.count()} valores inválidos")

In [0]:
# validar ocupação
occupancy_columns = ["ttm_occupancy", "ttm_adjusted_occupancy", "l90d_occupancy", "l90d_adjusted_occupancy"]

for c in occupancy_columns:
    invalid_occupancy = listings.filter((col(c) < 0) | (col(c) > 1))
    print(f"{c}: {invalid_occupancy.count()} valores inválidos")

#### Organização final das colunas

In [0]:
from pyspark.sql.functions import col, when

listings = (
    listings
    .withColumn("guests_missing", when(col("guests").isNull(), True).otherwise(False))
    .withColumn("bedrooms_missing", when(col("bedrooms").isNull(), True).otherwise(False))
    .withColumn("cleaning_fee_missing", when(col("cleaning_fee").isNull(), True).otherwise(False))
    .withColumn("extra_guest_fee_missing", when(col("extra_guest_fee").isNull(), True).otherwise(False))
)

In [0]:
final_columns = [
    "listing_id", "listing_name", "listing_type", "room_type", "property_usage_type", "guests", "bedrooms", "beds", "baths", "min_nights", "latitude", "longitude", "exact_location", "host_id", "host_name", "superhost", "professional_management", "instant_book", "guest_favorite", "num_reviews", "rating_overall", "rating_accuracy", "rating_checkin", "rating_cleanliness", "rating_communication", "rating_location", "rating_value", "currency", "cleaning_fee", "extra_guest_fee", "single_fee_structure", "ttm_revenue", "ttm_avg_rate", "ttm_occupancy", "ttm_revpar", "ttm_avg_min_nights", "ttm_avg_length_of_stay", "ttm_reserved_days", "ttm_blocked_days", "ttm_available_days", "ttm_total_days", "l90d_revenue", "l90d_avg_rate", "l90d_occupancy", "l90d_revpar", "l90d_avg_min_nights", "l90d_avg_length_of_stay", "l90d_reserved_days", "l90d_blocked_days", "l90d_available_days", "l90d_total_days", "has_description", "has_photos", "guests_missing", "bedrooms_missing", "cleaning_fee_missing", "extra_guest_fee_missing", "capacity_info_available", "fee_info_available", "revenue_info_available"
]

listings_silver = listings.select(final_columns)
display(listings_silver)

In [0]:
print(f"Registros: {listings_silver.count()}")
print(f"Colunas: {len(listings_silver.columns)}")

In [0]:
display(
    listings_silver.select(
        "listing_id", "listing_name", "room_type", "property_usage_type", "guests", "bedrooms", "beds", "baths",
        "ttm_revenue", "ttm_avg_rate", "ttm_occupancy", "ttm_revpar", "capacity_info_available", "fee_info_available"
    )
)

#### Persistência da camada Silver

Após a aplicação das transformações e validações, os dados tratados
serão persistidos como uma tabela Delta na camada Silver.

A tabela Bronze permanece inalterada, preservando os dados originais
da fonte.

In [0]:
listings_silver.write.format("delta").mode("overwrite").saveAsTable("airbnb_joao_pessoa.silver.listings")

#### Validação pós escrita